# Solar flux over a day: RSTN one-second data, and ours

Plots the Sun at 1415 MHz across one UTC day from two places: the **Radio
Solar Telescope Network**'s one-second archive at NOAA, and **this telescope's
solar tracks** where it recorded one that day. Either can be binned to a
longer interval, or left raw.

RSTN is four US Air Force stations recording the Sun once a second at eight
fixed frequencies. Their local-noon values are what the Observe tab already
quotes beside ours; this is the full time series behind them. Two of the four
overlap a Glasgow afternoon - **San Vito** (Italy) and **Sagamore Hill**
(Massachusetts) - so our solar tracks can be compared with theirs sample for
sample, not only against a single noon number.

**The archive runs about a month behind.** NOAA posts each month roughly ten
days after it ends, so a solar track taken this week has nothing to compare
with until then. The last cell lists which of our solar tracks can be compared
now.

Runs from anywhere in the repository, in radioconda or the project `.venv`.
Downloaded RSTN files are kept in `~/.cache/srt_rstn/`, outside the repository
and the observatory's data.

In [ ]:
# Zoom and pan in the figures - drag a box to zoom, the arrows to pan, the house
# to go back. Needs ipympl in the kernel (it is in the project .venv); without
# it the figures come out static, and this cell says so.
INTERACTIVE = True

import gzip
import os
import re
import sys
import urllib.error
import urllib.request
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt


def _choose_backend(interactive):
    try:
        shell = get_ipython()
    except NameError:
        return 'none - not running in a notebook'
    if interactive:
        try:
            import ipympl  # noqa: F401
            shell.run_line_magic('matplotlib', 'widget')
            plt.rcParams['figure.dpi'] = 100
            return 'interactive (ipympl)'
        except Exception:
            print('ipympl is not installed in this kernel, so the figures are static; '
                  'pip install ipympl for zoom and pan')
    shell.run_line_magic('matplotlib', 'inline')
    plt.rcParams['figure.dpi'] = 120
    return 'static'


plt.rcParams['figure.figsize'] = (12, 4.5)
print('figures:', _choose_backend(INTERACTIVE))

In [ ]:
# Where things are - found from the notebook's own location, so it runs from
# the notebooks folder, the root, or anywhere with SRT_ROOT set.
def _find_root(start=None):
    here = Path(os.environ.get('SRT_ROOT') or start or Path.cwd()).resolve()
    for d in (here, *here.parents):
        if (d / 'receiver_scheduler' / 'h1_web_scheduler.py').exists():
            return d
    raise SystemExit('cannot find the repository: set SRT_ROOT to its path')

ROOT = _find_root()
SCHED = ROOT / 'receiver_scheduler'
OBS = SCHED / 'data' / 'observations'
if str(SCHED) not in sys.path:
    sys.path.insert(0, str(SCHED))   # drift_fit, scallop, rf_calibration, observatory

print(f'repository {ROOT}')
print(f'recordings {OBS}')

## Settings

`DATE = None` picks the day of our most recent solar track, so a new one shows
up without editing anything; give a date to look at another.

`BIN_S = None` (or `0`) plots the raw series - one second for RSTN, one record
for ours, usually 3 s. Anything else bins both onto the **same UTC bin edges**,
whole multiples of `BIN_S` since the epoch, so the two can be compared bin for
bin.

`STATISTIC` matters more than it looks. RSTN reports **whole SFU** - steps of
about 1.2% at 84 SFU. The mean of a bin averages through those steps; the
median of a bin of integers is itself an integer (or a half), so it keeps them.
Use `'median'` to shrug off a single-sample spike, `'mean'` to see below the
quantisation.

In [ ]:
DATE = None                    # the UTC day as 'YYYY-MM-DD'; None = our latest solar track

# RSTN stations to try. San Vito and Sagamore Hill overlap our afternoons;
# Learmonth and Palehua are on the far side of the Earth, and only matter for
# seeing the rest of the Sun's day.
STATIONS = ['san-vito', 'sagamore-hill', 'learmonth', 'palehua']
FREQ_MHZ = 1415                # one of 245 410 610 1415 2695 4995 8800 15400

BIN_S = 60                     # seconds per bin; None or 0 for raw
STATISTIC = 'mean'             # 'mean' or 'median' - see above
MIN_FILL = 0.5                 # drop bins holding less than this fraction of their samples
NORMALISE = False              # divide each series by its own median over our window

# Our telescope
REMOVE_SCALLOP = True          # take the tracking scallop out (scallop.py)
ABOVE_ATMOSPHERE = True        # correct to above the atmosphere, as RSTN quotes it
SKIP_WARMUP_S = 120            # drop the start of each recording: the gain is ~1% high (#42)

# RSTN cleaning. Samples at or below zero are always dropped; so are those below
# this fraction of the day's median - the ramp at sunrise and sunset, and the
# station's own calibration periods. Nothing is clipped from above: a burst is
# real, and exactly what RSTN exists to record.
RSTN_FLOOR = 0.3

CACHE = Path(os.environ.get('XDG_CACHE_HOME') or Path.home() / '.cache') / 'srt_rstn'

## RSTN: fetching and reading

One gzipped text file per station per day, a line a second:

```
LISS20260815104015      16      36     148      84     111     145     269     507
code+UTC timestamp     245     410     610    1415    2695    4995    8800   15400 MHz
```

Four things about the archive that the code has to allow for:

- **The lines are fixed-width, and a channel not recording is blank.** Reading
  by position is the only safe way: split on whitespace, and one quiet channel
  in the middle slides every later frequency into the wrong column.
- **A posted file need not contain 1415 MHz.** Sagamore Hill's 1415 channel was
  blank from mid-August 2026 (present on the 10th, empty on the 15th, 25th and
  31st) and in mid-June; Palehua's likewise. San Vito's has been dependable.
  The load below says which case it is.
- **Each station has its own file extension** - `.lis` at San Vito, `.k7o` at
  Sagamore Hill, `.phf` at Palehua - taken from the station code. So the file
  is found by reading the month's directory listing, not by building its name.
- **A station names its file by its own local date.** Sagamore Hill's evening
  runs past UTC midnight, so its file for 15 July ends at 00:08 UTC on the
  16th. A UTC day is assembled from the files either side as well and cut to
  the day by timestamp.

In [ ]:
RSTN_BASE = ('https://www.ngdc.noaa.gov/stp/space-weather/solar-data/'
             'solar-features/solar-radio/rstn-1-second')
RSTN_FREQS_MHZ = [245, 410, 610, 1415, 2695, 4995, 8800, 15400]
# Not strftime('%b'), which follows the locale.
_MONTHS = ['jan', 'feb', 'mar', 'apr', 'may', 'jun',
           'jul', 'aug', 'sep', 'oct', 'nov', 'dec']


def _get(url, timeout=60):
    req = urllib.request.Request(url, headers={'User-Agent': 'acre-road-srt-notebook'})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()


def rstn_listing(station, year, month):
    """File names in one station-month directory; [] if it is not there (yet)."""
    url = f'{RSTN_BASE}/{station}/{year:04d}/{month:02d}/'
    try:
        html = _get(url).decode('utf-8', 'replace')
    except urllib.error.HTTPError as err:
        if err.code == 404:
            return []
        raise
    return sorted(set(re.findall(r'href="(\d\d[a-z]{3}\d\d\.[a-z0-9]+\.gz)"', html)))


def rstn_file(station, day):
    """Local copy of one station's file for one of its own dates, or None."""
    stem = f'{day.day:02d}{_MONTHS[day.month - 1]}{day.year % 100:02d}.'
    folder = CACHE / station
    cached = sorted(folder.glob(stem + '*.gz')) if folder.exists() else []
    if cached:
        return cached[0]
    names = [n for n in rstn_listing(station, day.year, day.month) if n.startswith(stem)]
    if not names:
        return None
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / names[0]
    path.write_bytes(_get(f'{RSTN_BASE}/{station}/{day.year:04d}/{day.month:02d}/{names[0]}'))
    return path


def read_rstn(path, freq_mhz):
    """(unix seconds, SFU) for one frequency; NaN where the channel is blank.

    Each line is **fixed-width**: 18 characters of station code and UTC
    timestamp, then the eight frequencies in 8-character columns. A channel the
    station is not recording is left *blank*, so the columns have to be read by
    position. Splitting on whitespace instead would slide every later frequency
    one column to the left the moment one in the middle went quiet, and label
    them all wrongly without a word of complaint."""
    k = RSTN_FREQS_MHZ.index(freq_mhz)
    lo, hi = 18 + 8 * k, 26 + 8 * k
    t, v = [], []
    with gzip.open(path, 'rt', errors='replace') as fh:
        for line in fh:
            s = line[4:18]
            if not s.isdigit():
                continue
            try:
                t.append(datetime(int(s[:4]), int(s[4:6]), int(s[6:8]), int(s[8:10]),
                                  int(s[10:12]), int(s[12:14]), tzinfo=timezone.utc).timestamp())
            except ValueError:
                continue
            field = line[lo:hi].strip()
            try:
                v.append(float(field) if field else np.nan)
            except ValueError:
                v.append(np.nan)
    return np.array(t), np.array(v)


def day_bounds(day):
    t0 = datetime(day.year, day.month, day.day, tzinfo=timezone.utc).timestamp()
    return t0, t0 + 86400.0


def rstn_day(station, day, freq_mhz):
    """(t, SFU, files read, note) for one station over one UTC day, cleaned.
    `note` says why nothing came back, if nothing did - a file that is posted
    is not the same as a file with this frequency in it."""
    t0, t1 = day_bounds(day)
    ts, vs, files = [], [], []
    for off in (-1, 0, 1):
        path = rstn_file(station, day + timedelta(days=off))
        if path is None:
            continue
        t, v = read_rstn(path, freq_mhz)
        ts.append(t)
        vs.append(v)
        files.append(path.name)
    if not ts:
        return None, None, files, why_missing(day)
    t = np.concatenate(ts)
    v = np.concatenate(vs)
    t, first = np.unique(t, return_index=True)          # sorted, and no repeats
    v = v[first]
    keep = (t >= t0) & (t < t1)
    t, v = t[keep], v[keep].astype(float)
    if not len(t):
        return None, None, files, 'posted, but nothing inside this UTC day'
    if not np.isfinite(v).any():
        return None, None, files, (f'posted, but the {freq_mhz} MHz channel is blank - '
                                   f'the station recorded other frequencies, not this one')
    v[v <= 0] = np.nan
    if np.isfinite(v).any():
        v[v < RSTN_FLOOR * np.nanmedian(v)] = np.nan
    return t, v, files, ''


def why_missing(day):
    """The honest reason a day has no RSTN file."""
    nxt = date(day.year + (day.month == 12), day.month % 12 + 1, 1)
    expected = nxt + timedelta(days=10)
    if date.today() < expected + timedelta(days=5):
        return (f'not posted yet - NOAA posts each month about ten days after it '
                f'ends; expect it around {expected:%d %b %Y}')
    return 'no file: the station did not observe that day, or has stopped reporting'


def hhmm(ts):
    return datetime.fromtimestamp(ts, tz=timezone.utc).strftime('%H:%M')

In [ ]:
def latest_track_day():
    """The UTC day of our most recent solar track with a continuum product.
    Filenames are local time but still sort chronologically, so the newest is
    found by walking back from the end."""
    for p in sorted(OBS.glob('2*_track.h5'), reverse=True):
        try:
            try:
                hf = h5py.File(p, 'r')
            except OSError:
                hf = h5py.File(p, 'r', swmr=True)      # still being written
            with hf:
                if (str(hf.attrs.get('object_name', '')).lower() != 'sun'
                        or 'spectra_wide_kelvin' not in hf):
                    continue
                t = hf['timestamps'][:]
        except (OSError, KeyError):
            continue
        if len(t):
            return datetime.fromtimestamp(float(np.median(t)), tz=timezone.utc).date()
    return None


if DATE:
    day = datetime.strptime(DATE, '%Y-%m-%d').date()
else:
    day = latest_track_day() or datetime.now(timezone.utc).date()
    DATE = day.isoformat()
    print(f'DATE not set: using {DATE}, the day of our latest solar track')
rstn = {}
for st in STATIONS:
    t, v, files, note = rstn_day(st, day, FREQ_MHZ)
    if t is None or not np.isfinite(v).any():
        print(f'{st:14s} nothing - {note or "every sample below the floor"}')
        continue
    rstn[st] = (t, v)
    ok = np.isfinite(v)
    print(f'{st:14s} {ok.sum():6d} s kept of {len(v)}, {hhmm(t[ok][0])}-{hhmm(t[ok][-1])} UTC, '
          f'median {np.nanmedian(v):5.1f} SFU   ({", ".join(files)})')

## Ours

Every solar track that overlaps the UTC day, reduced to solar flux per record.
Recording filenames carry **local** time, so the files a day either side are
checked too and each recording's own UTC timestamps decide.

The reduction is the one the Observe tab's solar plot uses, and
`solar_flux_scallop.ipynb` walks it step by step: antenna temperature from the
file, the continuum band (no hydrogen, no LO spur), pilot bursts dropped, the
tracking scallop removed, the antenna theorem through the measured beam, and
the atmosphere taken off. It is left per record here, so that the binning
below is the only binning.

In [ ]:
import drift_fit
import rf_calibration
import scallop
from observatory import antenna_temperature_to_flux

K_TO_SFU = antenna_temperature_to_flux(1.0)     # linear, so one factor serves


def _open(path):
    try:
        return h5py.File(path, 'r')
    except OSError:
        return h5py.File(path, 'r', swmr=True)  # still being written


def our_tracks(day):
    """Solar-track recordings with any record inside the UTC day."""
    t0, t1 = day_bounds(day)
    found = []
    for off in (-1, 0, 1):
        stamp = (day + timedelta(days=off)).strftime('%Y%m%d')
        for p in sorted(OBS.glob(f'{stamp}_*_track.h5')):
            try:
                with _open(p) as hf:
                    if str(hf.attrs.get('object_name', '')).lower() != 'sun':
                        continue
                    t = hf['timestamps'][:]
            except (OSError, KeyError):
                continue
            if len(t) and t.max() >= t0 and t.min() < t1:
                found.append(p)
    return found


def our_flux(path):
    """(t, SFU per record, notes). Raises KeyError for a recording made before
    the fixed instrument, which has no continuum product to reduce."""
    with _open(path) as hf:
        a = dict(hf.attrs)
        if 'spectra_wide_kelvin' not in hf:
            raise KeyError('no calibrated continuum product (recorded before the fixed instrument)')
        f = hf['frequency_hz_wide'][:]
        spec = hf['spectra_wide_kelvin'][:]
        t = hf['timestamps'][:]
        burst = hf['pilot_burst'][:].astype(bool) if 'pilot_burst' in hf else np.zeros(len(t), bool)
    n = min(len(t), len(spec), len(burst))
    spec, t, burst = spec[:n], t[:n], burst[:n]
    keep = ~burst & (t >= t[0] + SKIP_WARMUP_S)
    spec, t = spec[keep], t[keep]
    _, _, band = drift_fit._band_window(a, f)
    t_a = np.nanmean(spec[:, band], axis=1)
    notes = [f'{a.get("gain_db")} dB']
    if float(a.get('gain_db', 0)) >= 40:
        notes.append('40 dB compresses the Sun ~6% (issue #35)')
    if REMOVE_SCALLOP:
        t_a, rep = scallop.correct(t_a, t, a)
        notes.append('scallop removed' if rep.get('ok') else 'scallop left in')
    sfu = t_a * K_TO_SFU
    if ABOVE_ATMOSPHERE:
        alt, _ = scallop.target_altaz(a, t, scallop.observer(a))
        trans = np.array([rf_calibration.atmospheric_transmission(x) for x in alt])
        ok = np.isfinite(trans) & (trans > 0.5)
        sfu = np.where(ok, sfu / np.where(ok, trans, 1.0), sfu)
        notes.append('above the atmosphere')
    t0, t1 = day_bounds(day)
    inday = (t >= t0) & (t < t1)
    return t[inday], sfu[inday], notes


ours = []
for p in our_tracks(day):
    try:
        t, s, notes = our_flux(p)
    except KeyError as err:
        print(f'{p.name}: skipped - {err}')
        continue
    ours.append((t, s))
    print(f'{p.name}: {len(t)} records, {hhmm(t[0])}-{hhmm(t[-1])} UTC, '
          f'median {np.median(s):5.1f} SFU  ({"; ".join(notes)})')
if not ours:
    print(f'no usable solar track of ours on {DATE}')

if ours:
    our_t = np.concatenate([x[0] for x in ours])
    our_s = np.concatenate([x[1] for x in ours])
    order = np.argsort(our_t)
    our_t, our_s = our_t[order], our_s[order]
    window = (our_t[0], our_t[-1])
else:
    our_t = our_s = None
    window = None

## Binning

Bins are whole multiples of `BIN_S` since the epoch, so a San Vito bin and one
of ours with the same centre cover exactly the same seconds. A bin holding less
than `MIN_FILL` of the samples it should - at the edge of a run, or across a
gap - is dropped rather than drawn from a handful of points.

In [ ]:
def binned(t, y, bin_s=BIN_S, statistic=STATISTIC):
    """(bin centres, values, counts). Raw - bin_s None or 0 - hands the finite
    samples straight back."""
    good = np.isfinite(y)
    t, y = t[good], y[good]
    if not bin_s or len(t) < 2:
        return t, y, np.ones(len(t), int)
    cadence = np.median(np.diff(t))
    idx = np.floor(t / bin_s).astype(np.int64)
    keys, start, counts = np.unique(idx, return_index=True, return_counts=True)
    if statistic == 'median':
        vals = np.array([np.median(y[s:s + c]) for s, c in zip(start, counts)])
    else:
        vals = np.add.reduceat(y, start) / counts
    full = counts >= MIN_FILL * bin_s / cadence
    return (keys[full] + 0.5) * bin_s, vals[full], counts[full]


def to_utc(ts):
    return [datetime.fromtimestamp(x, tz=timezone.utc) for x in ts]


def gapped(t, y, factor=2.5):
    """Break the line wherever two points are much further apart than the
    series' own spacing, so a gap - between two of our recordings, or a station
    dropping out - is drawn as a gap and not as a straight line through
    minutes nobody measured."""
    t = np.asarray(t, float)
    y = np.asarray(y, float)
    if len(t) < 3:
        return t, y
    step = np.median(np.diff(t))
    cut = np.flatnonzero(np.diff(t) > factor * step)
    if not len(cut):
        return t, y
    return (np.insert(t, cut + 1, t[cut] + step),
            np.insert(y, cut + 1, np.nan))


def scaled(t, y):
    """Each series over its own median in our window, if NORMALISE."""
    if not NORMALISE:
        return y
    sel = np.isfinite(y)
    if window is not None:
        sel &= (t >= window[0]) & (t <= window[1])
    if not sel.any():
        sel = np.isfinite(y)
    return y / np.median(y[sel])


def new_figure():
    fig, ax = plt.subplots()
    if hasattr(fig.canvas, 'header_visible'):
        fig.canvas.header_visible = False       # ipympl's 'Figure 1' strip
    return fig, ax


def robust_limits(ax, arrays, lo=0.5, hi=99.5):
    """Y limits from the bulk of the data, so one spike does not flatten the
    plot. Returns how many points fall outside, so nothing is hidden silently."""
    v = np.concatenate([a[np.isfinite(a)] for a in arrays if len(a)])
    if not len(v):
        return 0
    a, b = np.percentile(v, [lo, hi])
    pad = 0.08 * (b - a or abs(b) or 1.0)
    ax.set_ylim(a - pad, b + pad)
    return int(((v < a - pad) | (v > b + pad)).sum())


BIN_TEXT = 'raw' if not BIN_S else f'{BIN_S} s bins, {STATISTIC}'
Y_TEXT = 'relative to its own median' if NORMALISE else 'solar flux (SFU)'
STATION_STYLE = {'san-vito': 'C0', 'sagamore-hill': 'C2', 'learmonth': 'C4', 'palehua': 'C5'}

## The day

In [ ]:
fig, ax = new_figure()
drawn = []
for st, (t, v) in rstn.items():
    bt, bv, _ = binned(t, v)
    y = scaled(bt, bv)
    gt, gy = gapped(bt, y)
    ax.plot(to_utc(gt), gy, lw=0.6 if not BIN_S else 1.0, alpha=0.85,
            color=STATION_STYLE.get(st), label=f'RSTN {st}')
    drawn.append(y)
if ours:
    bt, bv, _ = binned(our_t, our_s)
    y = scaled(bt, bv)
    gt, gy = gapped(bt, y)
    ax.plot(to_utc(gt), gy, lw=1.4, color='C3', label='Acre Road')
    drawn.append(y)
    ax.axvspan(*to_utc(window), color='C3', alpha=0.07)

if drawn:
    off = robust_limits(ax, drawn)
    t0, t1 = day_bounds(day)
    ax.set_xlim(*to_utc((t0, t1)))
    ax.set_xlabel(f'UTC, {DATE}')
    ax.set_ylabel(Y_TEXT)
    ax.set_title(f'The Sun at {FREQ_MHZ} MHz, {DATE}  -  {BIN_TEXT}')
    ax.legend(fontsize=8, loc='best')
    import matplotlib.dates as mdates
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=timezone.utc))
    plt.show()
    if off:
        print(f'{off} points lie off the plot - spikes, or a burst; they are in the data'
              + (' (zoom out to see them)' if 'widget' in plt.get_backend() or 'ipympl' in plt.get_backend() else ''))
else:
    plt.close(fig)
    print('nothing to plot for this day')

In [ ]:
# Our window close up, with whatever RSTN recorded over the same minutes.
if ours:
    pad = 600.0
    lo, hi = window[0] - pad, window[1] + pad
    fig, ax = new_figure()
    drawn = []
    for st, (t, v) in rstn.items():
        sel = (t >= lo) & (t <= hi)
        if not np.isfinite(v[sel]).any():
            continue
        bt, bv, _ = binned(t[sel], v[sel])
        y = scaled(bt, bv)
        gt, gy = gapped(bt, y)
        ax.plot(to_utc(gt), gy, lw=0.8, alpha=0.9, color=STATION_STYLE.get(st), label=f'RSTN {st}')
        drawn.append(y)
    bt, bv, _ = binned(our_t, our_s)
    y = scaled(bt, bv)
    gt, gy = gapped(bt, y)
    ax.plot(to_utc(gt), gy, lw=1.4, color='C3', label='Acre Road')
    drawn.append(y)
    off = robust_limits(ax, drawn)
    ax.set_xlim(*to_utc((lo, hi)))
    ax.set_xlabel(f'UTC, {DATE}')
    ax.set_ylabel(Y_TEXT)
    ax.set_title(f'Our window, {hhmm(window[0])}-{hhmm(window[1])} UTC  -  {BIN_TEXT}')
    ax.legend(fontsize=8, loc='best')
    import matplotlib.dates as mdates
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=timezone.utc))
    plt.show()
    if not rstn:
        print(f'no RSTN data for {DATE} to compare - {why_missing(day)}')

## Numbers over our window

For each station that was observing while we were: its median against ours,
and whether the two move together. The second is the question worth asking -
whether the ~0.4% wobble left in a solar track after the scallop comes out is
the Sun or our receiver. If a station shows the same structure at the same
minutes, it is solar.

Both series are binned onto the same edges (60 s if `BIN_S` is raw) and a
straight line is taken off each before correlating, so a common slow drift -
the Sun setting through different airmass at the two sites - does not pass for
agreement.

In [ ]:
if not ours:
    print('no solar track of ours on this day, so nothing to compare')
elif not rstn:
    print(f'no RSTN data for {DATE} - {why_missing(day)}')
else:
    cmp_bin = BIN_S or 60
    ot, ov, _ = binned(our_t, our_s, bin_s=cmp_bin)
    print(f'{cmp_bin} s bins over {hhmm(window[0])}-{hhmm(window[1])} UTC; '
          f'ours median {np.median(ov):.1f} SFU\n')
    print(f'{"station":14s} {"bins":>5s} {"median":>7s} {"ours/it":>8s} {"r (detrended)":>14s}')
    for st, (t, v) in rstn.items():
        rt, rv, _ = binned(t, v, bin_s=cmp_bin)
        common, i_o, i_r = np.intersect1d(ot, rt, return_indices=True)
        if len(common) < 5:
            print(f'{st:14s} {len(common):5d}  not observing while we were')
            continue
        a, b = ov[i_o], rv[i_r]
        x = common - common.mean()
        da = a - np.polyval(np.polyfit(x, a, 1), x)
        db = b - np.polyval(np.polyfit(x, b, 1), x)
        r = np.corrcoef(da, db)[0, 1] if da.std() and db.std() else float('nan')
        print(f'{st:14s} {len(common):5d} {np.median(b):7.1f} {np.median(a) / np.median(b):8.3f} {r:+14.2f}')
    print('\nThe stations disagree with each other by 10-20 SFU on any day, so a ratio '
          'inside that is agreement.')

## What this can and cannot show

- **RSTN is built to catch bursts, not to do sub-percent photometry of the
  quiet Sun.** Whole-SFU values put a 1.2% step under every sample; a 60 s mean
  brings that to ~0.15% of pure quantisation, before the station's own noise.
  A correlation near zero with RSTN at the 0.5% level is weak evidence that a
  wobble is ours; a flare in both is unmistakable.
- **Absolute levels differ between stations by 10-20 SFU** on any day. Compare
  timing and shape, or set `NORMALISE = True`.
- **Ours carries the calibration that was in force when it was recorded.** The
  system temperature drifts several kelvin an hour, which on a ~1600 K Sun is a
  fraction of a percent - small next to the station spread, but not nothing.
- **Recordings at 40 dB (before 2026-09-15) compress the Sun by ~6%.** They are
  labelled as such above, and read low by about that much.

## Which of our solar tracks can be compared yet

Each UTC day we have a solar track on, against the two stations that overlap a
Glasgow afternoon: `ok` means NOAA has the file *and* it holds the chosen
frequency, `blank` that the file is there but the channel was not recording,
`not yet` that the month is not posted. Set `DATE` to an `ok` day to see both.
Files are fetched once and cached, so re-running this is quick.

In [ ]:
days = set()
for p in sorted(OBS.glob('2*_track.h5')):
    try:
        with _open(p) as hf:
            if str(hf.attrs.get('object_name', '')).lower() != 'sun' or 'spectra_wide_kelvin' not in hf:
                continue
            t = hf['timestamps'][:]
    except (OSError, KeyError):
        continue
    if len(t):
        days.add(datetime.fromtimestamp(float(np.median(t)), tz=timezone.utc).date())

def coverage(station, d):
    t, v, files, note = rstn_day(station, d, FREQ_MHZ)
    if t is not None and np.isfinite(v).any():
        return 'ok'
    # Judged by the reason, not by whether any file came back: the day either
    # side can be posted when the day itself is not.
    return 'blank' if 'blank' in note else 'not yet'

print(f'{FREQ_MHZ} MHz')
print(f'{"UTC day":12s} {"san-vito":>10s} {"sagamore-hill":>14s}')
for d in sorted(days):
    print(f'{d.isoformat():12s} {coverage("san-vito", d):>10s} {coverage("sagamore-hill", d):>14s}')